# Step 1: 

This file aims to synthesise variables independent of each another. 
Distribution of each variable is based on national statistcs and published papers using NPD/HES/ECHILD.

Vars to synthesise: 
- Pupil Matching Reference
- tokenid	
- month_of_birth
- gender
- ethnicity
- imd_deciles
- gestational_age
- regions of birth (by residence of mother)


In [ ]:
"""
Install packages
!pip install pandas
!pip install numpy
"""

In [38]:
# importing relevant packages
import os
import pandas as pd
import numpy as np
import random
import string

os.chdir(r"C:\Users\uctvjla\OneDrive - University College London\Desktop\ECHILD_Synthetic\Python")

In [2]:
# Set the seed for reproducibility
np.random.seed(1234)

# Number of people
n_people = 500000

In [3]:
## Define Function to convert from decimal to alternate base numbers

def decimal_to_alt_base(numbers, alt_base, num_length, digits):
    alt_vals = []
    for num in numbers:
        digit_vals = np.zeros(num_length, dtype=int)
        for i in range(num_length - 1, -1, -1):
            divisor = alt_base**i
            digit_vals[num_length - i] = num // divisor
            num -= digit_vals[num_length - i] * divisor
        
        alt_val = ''.join(digits[digit_vals + 1])
        alt_vals.append(alt_val)
    
    return alt_vals

In [4]:
# Function to generate IDs
def generate_ids(number, digits, id_length, prefix="", suffix=""):
    n_digits = len(digits)
    combinations = n_digits**id_length
    
    if combinations < number:
        raise ValueError("Not possible to generate the requested number of distinct IDs based on digits and id_length.")
    
    if combinations < np.iinfo(np.int32).max:
        sample_vals = np.random.choice(combinations, size=number, replace=False)
        sample_ids = decimal_to_alt_base(sample_vals, n_digits, id_length, digits)
    elif np.exp(- number**2 / 2 / combinations) > 0.999:
        # Random chance of generating a unique sequence is > 99.9%
        while True:
            sample_ids = [''.join(random.choices(digits, k=id_length)) for _ in range(number)]
            if len(set(sample_ids)) == number:
                break
    else:
        raise ValueError("Generating a random sequence based on these inputs is too difficult.")
    
    sample_ids = [prefix + id_ + suffix for id_ in sample_ids]
    return sample_ids

In [5]:
# Generate gender (50/50)
gender = np.random.choice([0, 1], n_people, replace=True)

In [6]:
# Generate ethnicity
# based on National Statistics
# https://www.ethnicity-facts-figures.service.gov.uk/uk-population-by-ethnicity/national-and-regional-populations/population-of-england-and-wales/latest

# Define ethnicity percentages
ethnicity_asian_bangleshi = 1.1
ethnicity_asian_chinese = 0.7
ethnicity_asian_indian = 3.1
ethnicity_asian_pakistani = 2.7
ethnicity_asian_other = 1.6
ethnicity_black_african = 2.5
ethnicity_black_carribean = 1.0
ethnicity_black_other = 0.5
ethnicity_mixed_white_asian = 0.8
ethnicity_mixed_white_black_african = 0.4
ethnicity_mixed_white_black_carribean = 0.9
ethnicity_mixed_other = 0.8
ethnicity_white_gypsie_or_irish_trav = 0.1
ethnicity_white_roma = 0.2
ethnicity_white_irish = 0.9
ethnicity_white_other = 6.2
ethnicity_other_arab = 0.6
ethnicity_other_any_other = 1.6

# Create a list of ethnicity objects
all_ethnicity_objects = ["asian_bangleshi", "asian_chinese", "asian_indian", "asian_pakistani", "asian_other",
                         "black_african", "black_carribean", "black_other", "mixed_white_asian",
                         "mixed_white_black_african", "mixed_white_black_carribean", "mixed_other",
                         "white_gypsie_or_irish_trav", "white_roma", "white_irish", "white_other",
                         "other_arab", "other_any_other"]

# Generate ethnicity data based on percentages
all_ethnicity_data = []
for i in all_ethnicity_objects:
    temp = np.repeat(i, int(n_people * globals()["ethnicity_" + i] / 100))
    all_ethnicity_data.extend(temp)

# Generate ethnicity data for "white_british"
ethnicity_white_british = np.repeat("white_british", n_people - len(all_ethnicity_data))
all_ethnicity_data.extend(ethnicity_white_british)

# Shuffle the ethnicity data
np.random.shuffle(all_ethnicity_data)

# Remove unnecessary variables
del all_ethnicity_objects, ethnicity_white_british


In [34]:
# Month of Birth (https://www.statista.com/chart/5814/the-months-of-the-year-with-the-most-births/)
# Average daily number of births per month (1995 - 2014, ONS)
month_of_birth_counts = {
    "jan": 54951,
    "feb": 51826,
    "mar": 53561,
    "apr": 53264,
    "may": 56071,
    "jun": 55027,
    "jul": 57754,
    "aug": 56773,
    "sep": 57035,
    "oct": 55169,
    "nov": 54054,
    "dec": 54754
}

# Calculate the total count
total_count = sum(month_of_birth_counts.values())

# Calculate the probability distribution
month_of_birth_probabilities = {key: value / total_count for key, value in month_of_birth_counts.items()}

# Generate month_of_birth_data based on probabilities
month_of_birth_data = np.random.choice(list(month_of_birth_probabilities.keys()), n_people, p=list(month_of_birth_probabilities.values()))

In [15]:
# Generate imd_deciles
imd_deciles = np.tile(np.arange(1, 11), n_people // 10)


In [16]:
# Gestational age distribution (source: https://www.bmj.com/content/371/bmj.m4075)
gestational_age_percentages = {
    "less_than_28": 0.2,
    "28_29": 0.2,
    "30_31": 0.3,
    "32": 0.3,
    "33": 0.4,
    "34": 0.7,
    "35": 1.1,
    "36": 2.3,
    "37": 5.3,
    "38": 13.5,
    "39": 22.7,
    "40": 28.3,
    "41": 20.6,
    "42": 4.3
}

# Create gestational age data based on percentages
gestational_age_data = []
for ga, percentage in gestational_age_percentages.items():
    temp = int(n_people * (percentage / 100))
    temp = np.repeat(ga, temp)
    gestational_age_data.extend(temp)

# Shuffle the gestational age data
np.random.shuffle(gestational_age_data)


In [30]:
# Resgor based on 2008 data
# Source: https://www.ons.gov.uk/peoplepopulationandcommunity/birthsdeathsandmarriages/livebirths/datasets/birthsbyareaofusualresidenceofmotheruk
resgor_weights = {
    "north_east": 4,
    "north_west": 13,
    "yorkshire_humber": 10,
    "east_midlands": 8,
    "west_midlands": 11,
    "east": 11,
    "london": 19,
    "south_east": 15,
    "south_west": 9
}

# Create a weighted list
weighted_resgors = [region for region, weight in resgor_weights.items() for _ in range(int(n_people * weight / 100))]

# Shuffle the list
np.random.shuffle(weighted_resgors)

# Add "resgor_scotland" and "resgor_wales" randomly
weighted_resgors.extend(["resgor_scotland"] * int(n_people * 0.005))
weighted_resgors.extend(["resgor_wales"] * int(n_people * 0.005))

# Shuffle the list again to distribute "scotland" and "wales" randomly
np.random.shuffle(weighted_resgors)

# Use the first n_people elements
all_resgor_data = weighted_resgors[:n_people]

In [18]:
# PMRS and Token IDs
pmrs = generate_ids(n_people, list(map(str, range(10))) + list(string.ascii_uppercase[:6]), 16, prefix="CCF")
token_ids = generate_ids(n_people, list(map(str, range(10))) + list(string.ascii_uppercase), 11, prefix="0000")


In [ ]:
# add vars to pandas dataframe
people = pd.DataFrame({
    'PupilMatchingRefAnonymous': pmrs,
    'tokenid': token_ids,
    'month_of_birth': np.random.choice(month_of_birth_data, n_people),
    'gender': np.random.choice(gender, n_people),
    'ethnicity': np.random.choice(all_ethnicity_data, n_people),
    'imd_deciles': np.random.choice(imd_deciles, n_people),
    'gestational_age': np.random.choice(gestational_age_data, n_people),
    'resgor_birth': np.random.choice(all_resgor_data, n_people)
})


In [39]:
# Data to CSV
people.to_csv("1_people_jupyter.csv", index = False)

for var_name in list(globals().keys()):
    if var_name != 'people':
        del globals()[var_name]


In [61]:
## Extra - comparison of csv generated by R vs Python (same seed)
import pandas as pd
people_r = pd.read_csv("1_people.csv")
people_jup = pd.read_csv("1_people_jupyter.csv")

print(people_r.describe())
print(people_jup.describe())


              gender    imd_deciles
count  500000.000000  500000.000000
mean        0.500464       5.500000
std         0.500000       2.872284
min         0.000000       1.000000
25%         0.000000       3.000000
50%         1.000000       5.500000
75%         1.000000       8.000000
max         1.000000      10.000000
              gender    imd_deciles
count  500000.000000  500000.000000
mean        0.500616       5.498542
std         0.500000       2.874087
min         0.000000       1.000000
25%         0.000000       3.000000
50%         1.000000       5.000000
75%         1.000000       8.000000
max         1.000000      10.000000


In [63]:
cross_table_r = pd.crosstab(people_r['ethnicity'], people_r['imd_deciles'], margins=True, margins_name='Total')
cross_table_jup = pd.crosstab(people_jup['ethnicity'], people_jup['imd_deciles'], margins=True, margins_name='Total')

print(cross_table_r)
print(cross_table_jup)

imd_deciles                      1      2      3      4      5      6      7  \
ethnicity                                                                      
asian_bangleshi                563    526    546    526    568    538    554   
asian_chinese                  345    356    356    334    342    347    364   
asian_indian                  1548   1525   1475   1602   1496   1568   1578   
asian_other                    828    829    854    798    806    830    690   
asian_pakistani               1377   1306   1375   1406   1314   1329   1366   
black_african                 1264   1248   1301   1184   1259   1229   1273   
black_carribean                467    504    527    502    537    527    506   
black_other                    272    238    222    256    237    249    258   
mixed_other                    393    407    374    385    422    399    387   
mixed_white_asian              398    420    382    426    354    402    429   
mixed_white_black_african      193    20